# E-GraphSAGE (BoT-IoT) — PyTorch Geometric Version
**Converted from DGL to PyG. Optimised for 8 GB RAM (no GPU required).**

Install once before running:
```
pip install torch torchvision torchaudio
pip install torch_geometric
pip install scikit-learn pandas numpy seaborn matplotlib
```

## Cell 1 — Imports

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

import pandas as pd
import numpy as np
import socket, struct, random
import gc

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.utils import class_weight

import seaborn as sns
import matplotlib.pyplot as plt

from torch_geometric.data import Data
from torch_geometric.nn import MessagePassing

# Always use CPU to protect your 8 GB RAM — change to 'cuda' only if you have a GPU
DEVICE = 'cpu'
print(f'Using device: {DEVICE}')
print(f'PyTorch version: {torch.__version__}')

## Cell 2 — Load & Clean Data

In [ ]:
# ── Load CSV ──────────────────────────────────────────────────────────────────
data = pd.read_csv('./bot.csv')

# Drop columns that were dropped in the original notebook
data.drop(columns=['subcategory','pkSeqID','stime','flgs','attack',
                   'state','proto','seq'], inplace=True, errors='ignore')

data.rename(columns={'category': 'label'}, inplace=True)

print('Label distribution (raw):')
print(data.label.value_counts())
print(f'\nTotal rows: {len(data):,}')

## Cell 3 — Subsample to Fit in RAM  
*(same sampling fractions as the original notebook)*

In [ ]:
# Subsample heavy classes — same as original notebook
DDoS           = data[data['label'] == 'DDoS'].sample(frac=0.1, random_state=42)
DoS            = data[data['label'] == 'DoS'].sample(frac=0.1, random_state=42)
Reconnaissance = data[data['label'] == 'Reconnaissance'].sample(frac=0.1, random_state=42)
Normal         = data[data['label'] == 'Normal']
Theft          = data[data['label'] == 'Theft']

data = pd.concat([DDoS, DoS, Reconnaissance, Normal, Theft]).reset_index(drop=True)

# Free the full DataFrame from memory immediately
del DDoS, DoS, Reconnaissance, Normal, Theft
gc.collect()

print('Label distribution (after sampling):')
print(data.label.value_counts())
print(f'\nTotal rows after sampling: {len(data):,}')

## Cell 4 — Encode Labels & Build Node IDs

In [ ]:
# Encode class labels to integers
le = LabelEncoder()
data['label'] = le.fit_transform(data['label'])
NUM_CLASSES = len(le.classes_)
print('Classes:', le.classes_)
print('Encoded as:', list(range(NUM_CLASSES)))

# Convert IPs/ports to strings for node ID building
data['saddr'] = data['saddr'].astype(str)
data['daddr'] = data['daddr'].astype(str)
data['sport'] = data['sport'].astype(str)
data['dport'] = data['dport'].astype(str)

# Randomise source IPs (same as original notebook)
data['saddr'] = data['saddr'].apply(
    lambda x: socket.inet_ntoa(struct.pack('>I', random.randint(0xac100001, 0xac1f0001)))
)

# Build node identifiers:  IP:port
data['saddr'] = data['saddr'] + ':' + data['sport']
data['daddr'] = data['daddr'] + ':' + data['dport']
data.drop(columns=['sport', 'dport'], inplace=True)

print(f'\nUnique source nodes: {data["saddr"].nunique():,}')
print(f'Unique dest nodes:   {data["daddr"].nunique():,}')

## Cell 5 — Feature Engineering & Normalisation

In [ ]:
# One-hot encode categorical columns (same as original)
cat_cols = [c for c in ['flgs_number', 'state_number', 'proto_number'] if c in data.columns]
if cat_cols:
    data = pd.get_dummies(data, columns=cat_cols)

# Replace inf / NaN
data.replace([np.inf, -np.inf], np.nan, inplace=True)
data.fillna(0, inplace=True)

# Identify feature columns (everything except src, dst, label)
feat_cols = [c for c in data.columns if c not in ['saddr', 'daddr', 'label']]
EDGE_DIM = len(feat_cols)
print(f'Edge feature dimension: {EDGE_DIM}')

# Normalise features
scaler = StandardScaler()
data[feat_cols] = scaler.fit_transform(data[feat_cols].values.astype(np.float32))

print(f'Data shape: {data.shape}')
data.head(3)

## Cell 6 — Train / Test Split

In [ ]:
X_train, X_test = train_test_split(
    data, test_size=0.3, random_state=42, stratify=data['label']
)
X_train = X_train.reset_index(drop=True)
X_test  = X_test.reset_index(drop=True)

print(f'Train flows: {len(X_train):,}')
print(f'Test flows:  {len(X_test):,}')

# Free full data from memory
del data
gc.collect()

## Cell 7 — Build PyG Data Objects
*(replaces the DGL `from_networkx` + `G.ndata` / `G.edata` setup)*

In [ ]:
def build_pyg_data(df: pd.DataFrame, feat_cols: list, edge_dim: int) -> Data:
    """
    Build a PyG Data object from a flow DataFrame.
    
    Equivalent to the original notebook's:
        G = nx.from_pandas_edgelist(...)
        G = from_networkx(G, edge_attrs=['h','label'])
        G.ndata['h'] = th.ones(G.num_nodes(), edge_dim)
    """
    # Map IP:port strings → integer node IDs
    all_nodes = pd.concat([df['saddr'], df['daddr']]).unique()
    node2id   = {n: i for i, n in enumerate(all_nodes)}
    num_nodes = len(all_nodes)

    src_ids = torch.tensor(df['saddr'].map(node2id).values, dtype=torch.long)
    dst_ids = torch.tensor(df['daddr'].map(node2id).values, dtype=torch.long)

    # Build directed edge_index  (shape: [2, num_edges])
    edge_index = torch.stack([src_ids, dst_ids], dim=0)

    # Edge features and labels
    edge_attr = torch.tensor(df[feat_cols].values, dtype=torch.float32)
    y         = torch.tensor(df['label'].values,    dtype=torch.long)

    # Node features = all-ones, dim == edge_dim  (paper Algorithm 1, line 1)
    x = torch.ones(num_nodes, edge_dim, dtype=torch.float32)

    return Data(x=x, edge_index=edge_index, edge_attr=edge_attr, y=y)


# Build train & test graphs
train_data = build_pyg_data(X_train, feat_cols, EDGE_DIM)
test_data  = build_pyg_data(X_test,  feat_cols, EDGE_DIM)

print(f'Train graph — nodes: {train_data.num_nodes:,}  edges: {train_data.num_edges:,}')
print(f'Test  graph — nodes: {test_data.num_nodes:,}  edges: {test_data.num_edges:,}')
print(f'Edge feature dim: {train_data.edge_attr.shape[1]}')

# Free DataFrames — done with them
del X_train, X_test
gc.collect()

## Cell 8 — Model Definition
*(replaces `SAGELayer`, `SAGE`, `MLPPredictor`, `Model` DGL classes)*

In [ ]:
# ── EGraphSAGEConv: single edge-aware conv layer ──────────────────────────────
class EGraphSAGEConv(MessagePassing):
    """
    PyG equivalent of the original SAGELayer.

    Message:    m_ij = W_msg( concat(h_i, e_ij) )
    Aggregate:  h_N(j) = MEAN(m_ij for i in N(j))
    Update:     h_j = ReLU( W_apply( concat(h_j, h_N(j)) ) )
    """
    def __init__(self, node_dim: int, edge_dim: int, out_dim: int):
        super().__init__(aggr='mean')                        # mean aggregation
        self.W_msg   = nn.Linear(node_dim + edge_dim, out_dim)
        self.W_apply = nn.Linear(node_dim + out_dim,  out_dim)
        self.norm    = nn.LayerNorm(out_dim)

    def forward(self, x, edge_index, edge_attr):
        # x          : [N, node_dim]
        # edge_index : [2, E]
        # edge_attr  : [E, edge_dim]
        h_neigh = self.propagate(edge_index, x=x, edge_attr=edge_attr)
        # h_neigh : [N, out_dim]
        out = self.W_apply(torch.cat([x, h_neigh], dim=-1))
        return self.norm(F.relu(out))

    def message(self, x_j, edge_attr):
        # x_j       : [E, node_dim]  — source node embeddings
        # edge_attr : [E, edge_dim]
        return self.W_msg(torch.cat([x_j, edge_attr], dim=-1))


# ── SAGE: two stacked conv layers ────────────────────────────────────────────
class SAGE(nn.Module):
    """
    PyG equivalent of original SAGE class (2 SAGELayer blocks).
    hidden_dim matches original (128).
    """
    def __init__(self, node_dim, edge_dim, out_dim, dropout=0.2):
        super().__init__()
        self.conv1   = EGraphSAGEConv(node_dim, edge_dim, 128)
        self.conv2   = EGraphSAGEConv(128,      edge_dim, out_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, edge_index, edge_attr):
        h = self.conv1(x,            edge_index, edge_attr)
        h = self.dropout(h)
        h = self.conv2(h,            edge_index, edge_attr)
        return h                          # [N, out_dim]


# ── MLPPredictor: edge classifier ────────────────────────────────────────────
class MLPPredictor(nn.Module):
    """
    PyG equivalent of original MLPPredictor.
    Concatenates src and dst node embeddings → linear → class logits.
    """
    def __init__(self, in_features, num_classes):
        super().__init__()
        self.W = nn.Linear(in_features * 2, num_classes)

    def forward(self, h, edge_index):
        src, dst = edge_index
        edge_emb = torch.cat([h[src], h[dst]], dim=-1)   # [E, in_features*2]
        return self.W(edge_emb)                           # [E, num_classes]


# ── Full Model ────────────────────────────────────────────────────────────────
class EGraphSAGE(nn.Module):
    """
    PyG equivalent of the original Model class.
    """
    def __init__(self, edge_dim, hidden_dim=128, num_classes=5, dropout=0.2):
        super().__init__()
        self.gnn  = SAGE(edge_dim, edge_dim, hidden_dim, dropout)
        self.pred = MLPPredictor(hidden_dim, num_classes)

    def forward(self, data: Data):
        x, edge_index, edge_attr = data.x, data.edge_index, data.edge_attr
        h = self.gnn(x, edge_index, edge_attr)       # [N, hidden_dim]
        return self.pred(h, edge_index)               # [E, num_classes]


# ── Accuracy helper ───────────────────────────────────────────────────────────
def compute_accuracy(pred_logits, labels):
    return (pred_logits.argmax(1) == labels).float().mean().item()


print('Model classes defined successfully.')

## Cell 9 — Instantiate Model & Loss
*(replaces the original class_weight + CrossEntropyLoss + `.cuda()` block)*

In [ ]:
# ── Class-weighted loss (handles imbalance, same as original) ─────────────────
train_labels_np = train_data.y.numpy()
cw = class_weight.compute_class_weight(
    'balanced',
    classes=np.unique(train_labels_np),
    y=train_labels_np
)
class_weights_tensor = torch.tensor(cw, dtype=torch.float32).to(DEVICE)
criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)
print('Class weights:', dict(zip(le.classes_, np.round(cw, 3))))

# ── Model ─────────────────────────────────────────────────────────────────────
model = EGraphSAGE(
    edge_dim    = EDGE_DIM,
    hidden_dim  = 128,          # same as original
    num_classes = NUM_CLASSES,
    dropout     = 0.2,          # same as original
).to(DEVICE)

optimizer = torch.optim.Adam(model.parameters())

# Move graph data to device
train_data = train_data.to(DEVICE)
test_data  = test_data.to(DEVICE)

total_params = sum(p.numel() for p in model.parameters())
print(f'\nModel parameters: {total_params:,}')
print(model)

## Cell 10 — Training Loop
*(replaces the original 8000-epoch DGL training loop)*

> **RAM tip:** The full graph is passed each epoch (standard PyG pattern).  
> Reduce `EPOCHS` or `hidden_dim` if you still run out of memory.

In [ ]:
EPOCHS = 500          # original used 8000; 500 is enough for convergence on CPU
LOG_EVERY = 50        # print every N epochs

train_losses = []
train_accs   = []

for epoch in range(1, EPOCHS + 1):
    model.train()
    optimizer.zero_grad()

    # Forward pass on the FULL training graph
    pred = model(train_data)                         # [num_train_edges, num_classes]
    loss = criterion(pred, train_data.y)
    loss.backward()
    optimizer.step()

    if epoch % LOG_EVERY == 0 or epoch == 1:
        acc = compute_accuracy(pred, train_data.y)
        train_losses.append(loss.item())
        train_accs.append(acc)
        print(f'Epoch {epoch:4d} | Loss: {loss.item():.4f} | Train Acc: {acc:.4f}')

print('\nTraining complete.')

## Cell 11 — Training Curve Plot

In [ ]:
epochs_logged = list(range(LOG_EVERY, EPOCHS + 1, LOG_EVERY))
if len(epochs_logged) < len(train_losses):
    epochs_logged = [1] + epochs_logged

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(epochs_logged[:len(train_losses)], train_losses, 'b-o', markersize=4)
ax1.set_title('Training Loss')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.grid(True)

ax2.plot(epochs_logged[:len(train_accs)], train_accs, 'g-o', markersize=4)
ax2.set_title('Training Accuracy')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy')
ax2.grid(True)

plt.tight_layout()
plt.show()

## Cell 12 — Evaluation on Test Graph

In [ ]:
import timeit

model.eval()
start = timeit.default_timer()

with torch.no_grad():
    test_pred_logits = model(test_data)              # [num_test_edges, num_classes]

elapsed = timeit.default_timer() - start
print(f'Inference time: {elapsed:.4f} seconds')

# Convert to numpy for sklearn metrics
test_pred_enc = test_pred_logits.argmax(1).cpu().numpy()
test_true_enc = test_data.y.cpu().numpy()

# Decode back to original class names
test_pred_labels = le.inverse_transform(test_pred_enc)
test_true_labels = le.inverse_transform(test_true_enc)

print('\n=== Classification Report ===')
print(classification_report(
    test_true_labels, test_pred_labels,
    target_names=np.unique(test_true_labels), digits=4
))

## Cell 13 — Confusion Matrix

In [ ]:
def plot_confusion_matrix(cm, target_names, title='Confusion Matrix',
                          cmap=None, normalize=False):
    import itertools
    accuracy = np.trace(cm) / float(np.sum(cm))
    misclass = 1 - accuracy

    if cmap is None:
        cmap = plt.get_cmap('Blues')

    plt.figure(figsize=(10, 8))
    plt.imshow(cm, interpolation='nearest', cmap=cmap)
    plt.title(title)
    plt.colorbar()

    tick_marks = np.arange(len(target_names))
    plt.xticks(tick_marks, target_names, rotation=45)
    plt.yticks(tick_marks, target_names)

    cm_plot = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis] if normalize else cm
    thresh = cm_plot.max() / 1.5 if normalize else cm_plot.max() / 2

    for i, j in itertools.product(range(cm.shape[0]), range(cm.shape[1])):
        val = "{:0.4f}".format(cm_plot[i, j]) if normalize else "{:,}".format(cm[i, j])
        plt.text(j, i, val, horizontalalignment='center',
                 color='white' if cm_plot[i, j] > thresh else 'black')

    plt.tight_layout()
    plt.ylabel('True label')
    plt.xlabel(f'Predicted label\naccuracy={accuracy:.4f}; misclass={misclass:.4f}')
    plt.show()


target_names = np.unique(test_true_labels)
cm = confusion_matrix(test_true_labels, test_pred_labels, labels=target_names)

plot_confusion_matrix(cm, target_names, title='Confusion Matrix (counts)', normalize=False)
plot_confusion_matrix(cm, target_names, title='Confusion Matrix (normalised)', normalize=True)

## Cell 14 — Save Edge Embeddings (optional, for UMAP visualisation)
*(replaces the original `np.save('emb_mul.npy', ...)` cells)*

In [ ]:
# Extract edge embeddings from the test graph for downstream visualisation
model.eval()
with torch.no_grad():
    h_nodes = model.gnn(
        test_data.x, test_data.edge_index, test_data.edge_attr
    )                                              # [N, hidden_dim]
    src, dst = test_data.edge_index
    edge_emb = torch.cat([h_nodes[src], h_nodes[dst]], dim=-1)  # [E, hidden_dim*2]

edge_emb_np = edge_emb.cpu().numpy()
np.save('emb_mul.npy', edge_emb_np)
print(f'Saved edge embeddings: {edge_emb_np.shape}  →  emb_mul.npy')

# Also save the raw test features for comparison
np.save('raw_test.npy', test_data.edge_attr.cpu().numpy())
print('Saved raw test edge features  →  raw_test.npy')

## Cell 15 — UMAP Visualisation (optional)
*(same as the original notebook — requires `pip install umap-learn`)*

In [ ]:
# Install if not present:  pip install umap-learn
try:
    import umap

    reducer  = umap.UMAP(n_components=2, random_state=42)
    emb_viz  = reducer.fit_transform(edge_emb_np)

    df_umap = pd.DataFrame(emb_viz, columns=['comp1', 'comp2'])
    df_umap['label'] = test_true_labels

    plt.figure(figsize=(8, 8))
    sns.scatterplot(x='comp1', y='comp2', data=df_umap,
                    hue='label', palette='tab10', s=10, alpha=0.7)
    plt.title('UMAP of Edge Embeddings')
    plt.legend(loc='upper left', frameon=False)
    plt.tight_layout()
    plt.show()

except ImportError:
    print('umap-learn not installed. Run:  pip install umap-learn')

## Cell 16 — Save / Load Model Checkpoint

In [ ]:
# Save
torch.save(model.state_dict(), 'egraphsage_botiot.pt')
print('Model saved to egraphsage_botiot.pt')

# To reload later:
# model2 = EGraphSAGE(edge_dim=EDGE_DIM, hidden_dim=128,
#                     num_classes=NUM_CLASSES, dropout=0.2)
# model2.load_state_dict(torch.load('egraphsage_botiot.pt'))
# model2.eval()